# Evaluating the generator with deepeval

Stage 2 of the eval suite: the answers. The real agent answers each question in `evals/goldens/retriever_goldens.json`
as the chatbot would: `deepseek-flash` writes its own searches, reads the chunks and charts they return, and writes the
answer with page citations. deepeval then scores each answer with two metrics, judged by `deepseek-v4-pro`, a larger
model than the one that wrote the answers:

| Metric | What it asks | What it reads |
|---|---|---|
| Faithfulness | Is every claim in the answer supported by the chunks the agent retrieved? | The answer and the retrieved chunks |
| Answer Relevancy | Does each statement in the answer address the question? | The question and the answer |

`deepseek-v4-pro` reads text only, so a retrieved chart reaches the judge as its description, without the image the
answering model also saw. Neither metric needs a reference answer, so the golden set's reference answers are not used here.

The answers and the chunks behind them are saved in `generator_answers.json`, next to this notebook, so the same answers
can be scored again, by another judge for example, without running the agent. Delete that file to answer again.

A run makes 15 agent runs (2 to 4 `deepseek-flash` calls and 1 to 5 Jina searches each) and about 100 `deepseek-v4-pro`
calls. LangSmith tracing is off.

In [1]:
import json
import os
import sys
import time
import urllib.request
from pathlib import Path

# Set before deepeval and the project's config are imported, since both read them then
os.environ["DEEPEVAL_TELEMETRY_OPT_OUT"] = "1"  # deepeval sends no usage data
os.environ["DEEPEVAL_PER_TASK_TIMEOUT_SECONDS_OVERRIDE"] = "600"  # the reasoning judge can take minutes over a long context
os.environ["LANGSMITH_TRACING"] = "false"  # keeps the eval's agent runs out of the LangSmith project

PROJECT_ROOT = Path.cwd().parents[1]  # this notebook lives in evals/notebooks/
sys.path.insert(0, str(PROJECT_ROOT))

from deepeval import evaluate
from deepeval.evaluate.configs import CacheConfig, DisplayConfig, ErrorConfig
from deepeval.metrics import AnswerRelevancyMetric, FaithfulnessMetric
from deepeval.models import DeepSeekModel
from deepeval.test_case import LLMTestCase
from IPython.display import Markdown, display

from multimodal_rag import config
from multimodal_rag.generate import answer_question, build_agent  # importing it opens the vector store

GOLDENS_PATH = PROJECT_ROOT / "evals" / "goldens" / "retriever_goldens.json"
ANSWERS_PATH = PROJECT_ROOT / "evals" / "notebooks" / "generator_answers.json"  # the agent's answers and their chunks

# Judge model, on DeepSeek's own API. Its key is DEEPSEEK_API_KEY in .env.
JUDGE_MODEL = "deepseek-v4-pro"  # reads text only


def deepseek_balance():
    """The DeepSeek account's balance in USD, to the cent."""
    request = urllib.request.Request(
        config.DEEPSEEK_BASE_URL + "/user/balance",
        headers={"Authorization": "Bearer " + os.environ["DEEPSEEK_API_KEY"]},
    )
    return float(json.load(urllib.request.urlopen(request))["balance_infos"][0]["total_balance"])


print("Answering model:", config.GENERATION_MODEL, "| judge:", JUDGE_MODEL)

Answering model: deepseek-flash | judge: deepseek-v4-pro


## 1. Answer

Each question goes to `answer_question()`, the same function the test questions in `generate.py` use. Its result holds
the answer, the agent's searches, and every chunk those searches returned, each listed once.

In [2]:
goldens = json.loads(GOLDENS_PATH.read_text(encoding="utf-8"))
balance_before = deepseek_balance()

if ANSWERS_PATH.exists():
    answers = json.loads(ANSWERS_PATH.read_text(encoding="utf-8"))
    print(f"Read {len(answers)} saved answers from {ANSWERS_PATH.name}; delete it to answer again")
else:
    agent = build_agent()
    answers = []
    start = time.perf_counter()
    for number, golden in enumerate(goldens, start=1):
        answers.append(answer_question(agent, golden["input"]))
        print(f"{number:2}. answered after {len(answers[-1]['search_queries'])} searches")
    ANSWERS_PATH.write_text(json.dumps(answers, indent=1, ensure_ascii=False), encoding="utf-8")
    print(f"Answered {len(answers)} questions in {time.perf_counter() - start:.0f} seconds and saved them to {ANSWERS_PATH.name}")

 1. answered after 2 searches


 2. answered after 2 searches


 3. answered after 2 searches


 4. answered after 2 searches


 5. answered after 1 searches


...

11. answered after 2 searches


12. answered after 1 searches


13. answered after 2 searches


14. answered after 75 searches


15. answered after 2 searches
Answered 15 questions in 149 seconds and saved them to generator_answers.json


## 2. Test cases

Each answer becomes a test case with the chunks behind it, in the order the searches returned them. Below each question
are the agent's searches and the pages they reached, with the golden set's expected pages for comparison. That
comparison is not scored: it shows whether the agent's own searches found what stage 1's raw questions missed.

In [3]:
test_cases = []
for number, (golden, answer) in enumerate(zip(goldens, answers), start=1):
    test_cases.append(
        LLMTestCase(
            name=f"golden-{number}",
            input=answer["question"],
            actual_output=answer["answer"],
            retrieval_context=[chunk["text"] for chunk in answer["chunks"]],  # a chart as its description: the judge reads text only
        )
    )
    expected = golden["additional_metadata"]["pdf_pages"]
    pages = sorted({chunk["page_number"] for chunk in answer["chunks"]})
    found = [page for page in expected if page in pages]
    print(f"{number:2}. {len(answer['chunks'])} chunks from pages {pages}; expected {expected}, found {found or 'none'}")
    for query in answer["search_queries"]:
        print(f"      searched: {query}")

 1. 5 chunks from pages [51, 99, 102, 149, 158]; expected [149], found [149]
      searched: Groq agreement December 2025
      searched: Groq license technology acquisition
 2. 3 chunks from pages [121, 131, 164]; expected [121], found [121]
      searched: share repurchase authorization remaining amount
      searched: share repurchase program board of directors authorized additional shares billion
 3. 5 chunks from pages [23, 125, 126, 128, 129]; expected [125], found [125]
      searched: gross margin decrease fiscal 2026
      searched: gross profit margin discussion reasons
 4. 4 chunks from pages [49, 88, 99, 129]; expected [99], found [99]
      searched: number of employees workforce
      searched: employees in research and development
 5. 3 chunks from pages [20, 25, 30]; expected [20], found [20]
      searched: 2026 annual meeting of shareholders date time place record date who can vote
 6. 3 chunks from pages [78, 79, 135]; expected [78], found [78]
      searched: audito

## 3. Score

deepeval runs both metrics on every test case, several at a time. Each metric also writes the reason for its score.

In [4]:
judge = DeepSeekModel(model=JUDGE_MODEL, api_key=os.environ["DEEPSEEK_API_KEY"])
metrics = [FaithfulnessMetric(model=judge), AnswerRelevancyMetric(model=judge)]

start = time.perf_counter()
result = evaluate(
    test_cases=test_cases,
    metrics=metrics,
    display_config=DisplayConfig(show_indicator=False, print_results=False, inspect_after_run=False),
    cache_config=CacheConfig(write_cache=False),
    error_config=ErrorConfig(ignore_errors=True),  # a failed judge call shows as an error below instead of stopping the run
)
seconds = time.perf_counter() - start
balance_after = deepseek_balance()

scores = {test_result.name: {data.name: data for data in test_result.metrics_data} for test_result in result.test_results}
all_metric_data = [data for by_name in scores.values() for data in by_name.values()]
input_tokens = sum(data.input_tokens or 0 for data in all_metric_data)
output_tokens = sum(data.output_tokens or 0 for data in all_metric_data)
print(f"Scored {len(result.test_results)} test cases in {seconds:.0f} seconds")
print(f"Judge tokens: {input_tokens:,} input, {output_tokens:,} output")
print(f"DeepSeek balance: ${balance_before:.2f} before answering, ${balance_after:.2f} after scoring")

⚠ WARNING: No hyperparameters logged.
» ]8;id=14458081;https://deepeval.com/docs/evaluation-prompts\Log hyperparameters]8;;\ to attribute prompts and models to your test runs.

================================================================================

✓ Evaluation completed 🎉! (time taken: 106.5s | token cost: None)
» Test Results (15 total tests):
   » Pass Rate: 100.0% | Passed: 15 | Failed: 0

 ================================================================================ 

» Want to share evals with your team, or a place for your test cases to live? ❤️ 🏡
  » Run 'deepeval view' to analyze and save testing results on Confident AI.

Scored 15 test cases in 107 seconds
Judge tokens: 0 input, 0 output
DeepSeek balance: $1.81 before answering, $1.79 after scoring


## 4. Results

Scores run from 0 to 1. "Expected page found" says whether the agent's searches reached the golden set's pages.

In [5]:
def score_text(data):
    return "error" if data.score is None else f"{data.score:.2f}"


rows = [
    "| # | Content | Question | Searches | Expected page found | Faithfulness | Answer Relevancy |",
    "|---|---|---|---|---|---|---|",
]
by_content = {}  # content type -> list of (faithfulness, relevancy)
for number, (golden, answer) in enumerate(zip(goldens, answers), start=1):
    metadata = golden["additional_metadata"]
    pages = {chunk["page_number"] for chunk in answer["chunks"]}
    found = [page for page in metadata["pdf_pages"] if page in pages]
    found_text = "yes" if len(found) == len(metadata["pdf_pages"]) else ("partly" if found else "no")
    faithfulness = scores[f"golden-{number}"]["Faithfulness"]
    relevancy = scores[f"golden-{number}"]["Answer Relevancy"]
    rows.append(
        f"| {number} | {metadata['content']} | {golden['input']} | {len(answer['search_queries'])} | {found_text} "
        f"| {score_text(faithfulness)} | {score_text(relevancy)} |"
    )
    by_content.setdefault(metadata["content"], []).append((faithfulness.score, relevancy.score))
display(Markdown("\n".join(rows)))


def mean(values):
    values = [value for value in values if value is not None]
    return sum(values) / len(values) if values else float("nan")


summary = ["| Content | Questions | Mean faithfulness | Mean answer relevancy |", "|---|---|---|---|"]
for content, pairs in by_content.items():
    summary.append(f"| {content} | {len(pairs)} | {mean(f for f, _ in pairs):.2f} | {mean(r for _, r in pairs):.2f} |")
all_pairs = [pair for pairs in by_content.values() for pair in pairs]
summary.append(f"| **all** | {len(all_pairs)} | **{mean(f for f, _ in all_pairs):.2f}** | **{mean(r for _, r in all_pairs):.2f}** |")
display(Markdown("\n".join(summary)))

| # | Content | Question | Searches | Expected page found | Faithfulness | Answer Relevancy |
|---|---|---|---|---|---|---|
| 1 | text | What did NVIDIA get from its December 2025 agreement with Groq, and how much did it pay? | 2 | yes | 1.00 | 0.90 |
| 2 | text | How much room does NVIDIA still have to buy back its own shares? | 2 | yes | 1.00 | 1.00 |
| 3 | text | Why did NVIDIA's gross margin go down in fiscal 2026? | 2 | yes | 1.00 | 1.00 |
| 4 | text | How big is NVIDIA's workforce, and how many people work in research? | 2 | yes | 1.00 | 0.50 |
| 5 | text | When and where is the 2026 annual shareholder meeting, and who can vote at it? | 1 | yes | 1.00 | 1.00 |
| 6 | table | What did NVIDIA pay its auditor PwC in fiscal 2026, and how much of that was for the audit itself? | 1 | yes | 1.00 | 1.00 |
| 7 | table | How much of NVIDIA's Data Center revenue came from compute and how much from networking? | 1 | yes | 1.00 | 1.00 |
| 8 | table | How much revenue came from customers based in China, and how did that change from the year before? | 2 | yes | 1.00 | 0.75 |
| 9 | table | How much were NVIDIA's non-employee directors paid for fiscal 2026? | 2 | yes | 1.00 | 1.00 |
| 10 | table | Who are NVIDIA's largest shareholders? | 2 | yes | 1.00 | 1.00 |
| 11 | chart | Which organizations publish the most open-source AI repositories, according to the report? | 2 | yes | 1.00 | 0.89 |
| 12 | chart | How much cash has NVIDIA returned to shareholders over the past one, three and five years? | 1 | yes | 1.00 | 1.00 |
| 13 | chart | How is the CEO's target pay for fiscal 2026 split between salary, cash and stock awards? | 2 | yes | 1.00 | 0.67 |
| 14 | multi_page | Who chairs each of the Board's committees, and how often did each committee meet in fiscal 2026? | 75 | yes | 1.00 | 1.00 |
| 15 | multi_page | How much did NVIDIA hold in cash equivalents and marketable securities at the end of fiscal 2026, compared with a year earlier? | 2 | partly | 1.00 | 0.79 |

| Content | Questions | Mean faithfulness | Mean answer relevancy |
|---|---|---|---|
| text | 5 | 1.00 | 0.88 |
| table | 5 | 1.00 | 0.95 |
| chart | 3 | 1.00 | 0.85 |
| multi_page | 2 | 1.00 | 0.89 |
| **all** | 15 | **1.00** | **0.90** |

## 5. Reasons

The judge's reason for every score below 1, and any error.

In [6]:
for number in range(1, len(goldens) + 1):
    for name, data in scores[f"golden-{number}"].items():
        if data.score is None or data.score < 1:
            print(f"{number:2}. {name} {score_text(data)}: {data.error or data.reason}\n")

 1. Answer Relevancy 0.90: The score is 0.90 because the answer largely covers what NVIDIA acquired and the payment amount, but it includes a statement about the risk of incorporating technology, which is not directly relevant to the requested details. This minor irrelevant addition prevents a perfect score, while the core information remains accurate and on-topic, justifying the high relevance.

 4. Answer Relevancy 0.50: The score is 0.50 because the output addressed the core question about NVIDIA's workforce size and research count but included irrelevant details about technical role percentage, educational attainment, and turnover rate that were not asked, which lowered the relevancy from a perfect score.

 8. Answer Relevancy 0.75: The score is 0.75 because the output provides the requested China revenue and year-over-year change, but includes irrelevant statements about total company revenue and revenue from customers outside the United States, which are not specific to China and

## 6. Answers

What the agent answered, with its scores.

In [7]:
for number, answer in enumerate(answers, start=1):
    by_name = scores[f"golden-{number}"]
    display(Markdown(
        f"**{number}. {answer['question']}**\n\n{answer['answer']}\n\n"
        f"*Faithfulness {score_text(by_name['Faithfulness'])}, Answer Relevancy {score_text(by_name['Answer Relevancy'])}*\n\n---"
    ))

**1. What did NVIDIA get from its December 2025 agreement with Groq, and how much did it pay?**

**What NVIDIA obtained (page 149):** Under a **non-exclusive license agreement with Groq, Inc. entered into in December 2025**, NVIDIA licensed Groq's **language processing unit (LPU) technology** and **hired certain Groq employees**. Notably, **no customer contracts, existing products, or equity interests were purchased**.

**What it paid / total consideration (page 149):** Total consideration was **$17.0 billion**, consisting of:
- **$13.0 billion paid at closing**, and
- **$4 billion** (inclusive of imputed interest) **payable within one year**, recorded in Accrued and Other Current Liabilities.

**How it was allocated (page 149):** NVIDIA recorded **$14.4 billion of goodwill** (attributable primarily to the workforce and future development of the licensed technology, in the Compute & Networking reporting unit, and tax deductible) and a **$2.5 billion developed technology intangible asset**, valued using a cost-to-recreate methodology with a **five-year useful life**.

The report also notes the arrangement required "significant, nonrefundable payments" and that incorporating the licensed technology into NVIDIA's architectures and product roadmaps carries execution risk (page 102).

*Faithfulness 1.00, Answer Relevancy 0.90*

---

**2. How much room does NVIDIA still have to buy back its own shares?**

**As of January 25, 2026, NVIDIA was authorized to repurchase up to $58.5 billion of its common stock** (pages 121, 131, 164).

Context on how that figure came about:
- On August 26, 2025, the Board of Directors approved an **additional $60.0 billion** in share repurchase authorization, with no expiration date (page 121, 131, 164).
- In fiscal year 2026, NVIDIA repurchased **282 million shares for $40.4 billion** (page 131, 164).
- From **January 26, 2026 through February 20, 2026**, the company repurchased an additional **8 million shares for $1.5 billion** under a pre-established trading plan — which implies roughly $57.0 billion of authorization would remain after that date, though the report states only the $58.5 billion figure as of January 25, 2026 (page 131, 164).

The program may be suspended at any time at the company's discretion, and repurchases are subject to market conditions, applicable legal requirements, and other factors (page 121, 131).

*Faithfulness 1.00, Answer Relevancy 1.00*

---

**3. Why did NVIDIA's gross margin go down in fiscal 2026?**

NVIDIA's gross margin fell to **71.1% in fiscal 2026 from 75.0% in fiscal 2025**, a decline of 3.9 percentage points (page 125, page 128). The report cites two drivers:

1. **Business model transition** — the shift from offering Hopper HGX systems to Blackwell full-scale datacenter solutions (page 129, page 125).
2. **A $4.5 billion charge** in the first quarter of fiscal 2026 associated with H20 excess inventory and purchase obligations (page 129, page 125).

Additional context on inventory effects: total provisions for inventory and excess inventory purchase obligations were $7.2 billion in fiscal 2026 versus $3.7 billion in fiscal 2025, partly offset by releases of $1.5 billion and $689 million, respectively. The net effect of inventory provisions and sales of previously written-down items was an unfavorable impact on gross margin of 2.6% in fiscal 2026 versus 2.3% in fiscal 2025 (page 129, page 126).

*Faithfulness 1.00, Answer Relevancy 1.00*

---

**4. How big is NVIDIA's workforce, and how many people work in research?**

As of the end of fiscal year 2026, NVIDIA had approximately **42,000 employees** across 38 countries. Of those, about **31,000 were engaged in research and development**, while roughly 11,000 worked in sales, marketing, operations, and administrative positions (page 99).

The report also notes that more than 80 percent of the workforce hold technical roles and more than half hold an advanced degree, with a fiscal 2026 turnover rate of 3.7 percent (page 99).

*Faithfulness 1.00, Answer Relevancy 0.50*

---

**5. When and where is the 2026 annual shareholder meeting, and who can vote at it?**

NVIDIA’s 2026 Annual Meeting of Stockholders will be held **Wednesday, June 24, 2026, at 9:00 a.m. Pacific Time**, virtually at **www.virtualshareholdermeeting.com/NVDA2026** (pages 20, 25).

Stockholders of record at the close of business on **April 27, 2026** are entitled to vote (pages 20, 25).

*Faithfulness 1.00, Answer Relevancy 1.00*

---

...

**11. Which organizations publish the most open-source AI repositories, according to the report?**

According to the chart "NVIDIA is World's Largest Contributor to Open-Source AI" on page 7 (source: Hugging Face), repository counts by company grew from zero in January 2025 to these levels by February 2026:

| Company | Repos (Feb 2026) |
|---|---|
| NVIDIA | 420 |
| Alibaba Cloud | 340 |
| Hugging Face | 210 |
| AI2 | ~200 (shown in chart only, not in the table) |
| Meta | 150 |
| Google | 130 |
| IBM | 120 |
| ByteDance | 110 |
| Baidu | 100 |
| Tencent | 85 |
| Microsoft | 80 |
| Z.ai | 45 |

NVIDIA is the largest publisher of open-source AI repositories, followed by Alibaba Cloud, with Hugging Face, AI2, Meta, Google and IBM forming the next tier (page 7).

*Faithfulness 1.00, Answer Relevancy 0.89*

---

**12. How much cash has NVIDIA returned to shareholders over the past one, three and five years?**

NVIDIA returned the following amounts of capital to shareholders (page 24):

| Period | Dividends | Stock Repurchases | Total Returned |
|---|---|---|---|
| 1-Year | $1.0B | $40.4B | **$41.4 billion** |
| 3-Year | $2.2B | $84.1B | **$86.3 billion** |
| 5-Year | $3.0B | $94.2B | **$97.2 billion** |

So the vast majority came from stock repurchases rather than dividends. For context, in fiscal 2026 alone NVIDIA repurchased 282 million shares for $40.4 billion and paid $974 million in cash dividends (page 164).

*Faithfulness 1.00, Answer Relevancy 1.00*

---

**13. How is the CEO's target pay for fiscal 2026 split between salary, cash and stock awards?**

For fiscal 2026, CEO Jen-Hsun Huang's target pay totals **$35.5 million**, split as follows (page 65):

| Component | Target ($) | Share of target pay |
|---|---|---|
| Base Salary | 1,500,000 | 4% |
| Variable Cash | 3,000,000 | 8% |
| **Total Cash** | **4,500,000** | **~12%** |
| SY PSUs | 15,500,000 | 44% |
| MY PSUs | 15,500,000 | 44% |
| **Total Equity** | **31,000,000** | **~88%** |
| **TOTAL** | **35,500,000** | **100%** |

So roughly $4.5 million (12.7%) is cash — $1.5 million salary plus a $3.0 million variable cash target — and $31.0 million (87.3%) is stock awards, split evenly between SY PSUs and MY PSUs (pages 57, 65).

Notes from the report:
- Both salary and variable cash were flat with fiscal 2025, while equity rose $3.5 million, or 13%, from the fiscal 2025 target (page 65).
- Targets assume Base Compensation Plan level achievement; actual payouts differ. For fiscal 2026, revenue above the Stretch goal produced a 200% variable cash payout ($6,000,000), and Non-GAAP operating income above the Stretch goal made 150% of the SY PSU target (179,411 shares) eligible to vest (page 65).
- Over 90% of the CEO's total target pay is performance-based and at-risk, and 100% of his equity awards are PSUs (page 27).

*Faithfulness 1.00, Answer Relevancy 0.67*

---

**14. Who chairs each of the Board's committees, and how often did each committee meet in fiscal 2026?**

The committee chairs are:

- **Audit Committee:** A. Brooke Seawell, Chairperson (pages 26, 43)
- **Compensation Committee:** Dawn Hudson, Chairperson (page 45)
- **Nominating and Corporate Governance Committee:** Stephen C. Neal, Chairperson (page 45)

**Meetings in fiscal 2026:**
- **Compensation Committee:** met **three times** (page 45)
- **Nominating and Corporate Governance Committee:** met **three times** (page 45)

The retrieved report text does **not state the Audit Committee’s number of meetings**, so I don’t have enough information to give that figure.

*Faithfulness 1.00, Answer Relevancy 1.00*

---

**15. How much did NVIDIA hold in cash equivalents and marketable securities at the end of fiscal 2026, compared with a year earlier?**

**NVIDIA's cash, cash equivalents, and marketable securities (pages 130–131):**

| (In millions) | Jan 25, 2026 | Jan 26, 2025 | Change |
|---|---|---|---|
| Cash and cash equivalents | $10,605 | $8,589 | +$2,016 |
| Marketable securities | 51,951 | 34,621 | +$17,330 |
| **Total** | **$62,556** | **$43,210** | **+$19,346** |

So NVIDIA held **$62.6 billion** at the end of fiscal 2026, up from **$43.2 billion** a year earlier — an increase of about **$19.3 billion, or roughly 45%** (page 130). The MD&A liquidity discussion confirms "As of January 25, 2026, we had $62.6 billion in cash, cash equivalents, and marketable securities," with about $1.7 billion held outside the U.S. for which no related foreign or state taxes have been accrued (page 131).

One reconciliation note: Note 7 shows total cash equivalents and marketable securities with an estimated fair value of **$65.1 billion**, because it also includes **$4.8 billion** of long-term publicly-held equity securities (subject to lock-up restrictions through December 2027) that are reported within "Other Assets" rather than in cash/marketable securities on the balance sheet (page 153).

*Faithfulness 1.00, Answer Relevancy 0.79*

---